# Assignment 21: LangChain Document Loaders & Text Splitters

**Submitted By:** Abhishek Thakare

## Objective

The aim of this assignment is to practice loading TXT, CSV, PDF, directory, and web data using LangChain document loaders. I also compare different text splitting methods and combine loading and splitting in one function.

The assignment only focuses on data ingestion and chunking. I am not using a vector database or making any LLM calls.

## Input Files

The `data` folder contains the three mandatory sample files:

- `notes.txt` - text notes about document loading and chunking
- `data.csv` - sample employee data in tabular format
- `company_overview.pdf` - a small two-page company document

## Imports

In [1]:
import os
from urllib.parse import urlparse

from langchain_community.document_loaders import (
    TextLoader,
    CSVLoader,
    PyPDFLoader,
    DirectoryLoader,
    WebBaseLoader,
)
from langchain_text_splitters import (
    CharacterTextSplitter,
    RecursiveCharacterTextSplitter,
    MarkdownHeaderTextSplitter,
)

C:\Users\abhis\AppData\Local\Temp\ipykernel_13416\3774316921.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import (


USER_AGENT environment variable not set, consider setting it to identify your requests.


# Part 1 - Document Loaders in LangChain

## Task 1 - Text Loader

I am starting with `TextLoader` because `notes.txt` is a plain text file. I want to inspect how LangChain stores the text and metadata.

In [2]:
text_loader = TextLoader("data/notes.txt", encoding="utf-8")
text_documents = text_loader.load()

print("Number of documents loaded:", len(text_documents))
print("Document type:", type(text_documents[0]))
print("\nPage content preview:")
print(text_documents[0].page_content[:500])
print("\nMetadata:")
print(text_documents[0].metadata)

Number of documents loaded: 1
Document type: <class 'langchain_core.documents.base.Document'>

Page content preview:
Personal Knowledge Assistant - Project Notes

Introduction
This project aims to build a Personal Knowledge Assistant that can answer questions
from a variety of personal and company data sources. Before we can build any kind
of retrieval-augmented generation (RAG) system, we first need a reliable way to
ingest data from many different formats and turn it into clean, structured text
that a language model can reason over.

Why Document Loaders Matter
Real-world knowledge is scattered across many f

Metadata:
{'source': 'data/notes.txt'}


### Observation

Run the cell above and check the document count, content preview, and metadata. `TextLoader` keeps the loaded text in `page_content`, while information about the source is stored in `metadata`.

## Task 2 - CSV Loader

For the CSV file, I am using `CSVLoader`. I want to check whether each CSV row is converted into a separate LangChain `Document`.

In [3]:
csv_loader = CSVLoader(file_path="data/data.csv")
csv_documents = csv_loader.load()

print("Number of documents loaded:", len(csv_documents))
print("Document type:", type(csv_documents[0]))
print("\nSample document content:")
print(csv_documents[0].page_content)
print("\nSample metadata:")
print(csv_documents[0].metadata)

Number of documents loaded: 8
Document type: <class 'langchain_core.documents.base.Document'>

Sample document content:
employee_id: 101
name: Ananya Sharma
department: Engineering
role: Software Developer
years_experience: 3
location: Pune

Sample metadata:
{'source': 'data/data.csv', 'row': 0}


### Observation

The document count can be compared with the number of data rows in the CSV. The sample output shows that one row is represented as readable field-value text inside `page_content`.

## Task 3 - PDF Loader

I am using `PyPDFLoader` for the PDF file. I want to inspect the total number of pages and the text extracted from one page.

In [4]:
pdf_loader = PyPDFLoader("data/company_overview.pdf")
pdf_documents = pdf_loader.load()

print("Total pages:", len(pdf_documents))
print("Document type:", type(pdf_documents[0]))
print("\nSample page content:")
print(pdf_documents[0].page_content[:500])
print("\nPage metadata:")
print(pdf_documents[0].metadata)

Total pages: 2
Document type: <class 'langchain_core.documents.base.Document'>

Sample page content:
Company Overview Document
This document provides a short overview of our engineering practices and onboarding process for new
employees joining the Software Product Development team. It is intended to be used as sample
content for testing document loaders.
Section 1: Engineering Practices
Our engineering team follows clean code principles, structured requirement gathering through SRS
documents, and a well defined Software Development Life Cycle. Code reviews and automated testing
are mandatory b

Page metadata:
{'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-07-12T13:32:03+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-07-12T13:32:03+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': 'data/company_overview.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}


### Observation

`PyPDFLoader` returns page-level documents for this PDF. The page text is available in `page_content`, and page/source details are available in metadata.

## Task 4 - Directory Loader

The `data` folder contains TXT, CSV, and PDF files. Since these formats need different loaders, I use `DirectoryLoader` separately with the suitable loader class for each extension and combine the results.

In [5]:
loader_settings = {
    ".txt": (TextLoader, {"encoding": "utf-8"}),
    ".csv": (CSVLoader, {}),
    ".pdf": (PyPDFLoader, {}),
}

all_documents = []

for extension, (loader_class, loader_kwargs) in loader_settings.items():
    directory_loader = DirectoryLoader(
        "data",
        glob=f"**/*{extension}",
        loader_cls=loader_class,
        loader_kwargs=loader_kwargs,
    )
    loaded_docs = directory_loader.load()
    all_documents.extend(loaded_docs)

    print(f"{extension} documents loaded:", len(loaded_docs))
    print("Sources:", sorted(set(doc.metadata.get("source") for doc in loaded_docs)))
    print()

print("Total documents loaded from data folder:", len(all_documents))

loaded_extensions = {
    os.path.splitext(doc.metadata.get("source", ""))[1]
    for doc in all_documents
}
print("File types loaded:", sorted(loaded_extensions))

.txt documents loaded: 1
Sources: ['data\\notes.txt']

.csv documents loaded: 8
Sources: ['data\\data.csv']

.pdf documents loaded: 2
Sources: ['data\\company_overview.pdf']

Total documents loaded from data folder: 11
File types loaded: ['.csv', '.pdf', '.txt']


### Observation

The final `File types loaded` output is used to verify that `.txt`, `.csv`, and `.pdf` content was included. Different loader classes are still used because each file format stores data differently.

## Task 5 - WebBase Loader

I am using `WebBaseLoader` on a public webpage. The first 500 characters are printed to verify that textual content was extracted.

**Note:** This cell requires an internet connection.

In [6]:
web_url = "https://www.example.com/"
web_loader = WebBaseLoader(web_url)
web_documents = web_loader.load()

print("Number of web documents loaded:", len(web_documents))
print("\nFirst 500 characters:")
print(web_documents[0].page_content[:500])

Number of web documents loaded: 1

First 500 characters:
Example DomainExample DomainThis domain is for use in documentation examples without needing permission. Avoid use in operations.Learn more



### Observation

After running the cell with internet access, the first 500 characters should show text extracted from the public webpage instead of raw HTML tags.

# Part 2 - Text Splitters in LangChain

## Task 6 - Why Text Splitting is Required

### 1. Why can large documents not be directly passed to LLMs?

LLMs have a limit on how much text they can process at one time. A large PDF or long document may be bigger than this limit. Even when the complete document fits, sending unrelated sections can make the context less focused.

### 2. What problems does chunking solve in GenAI systems?

Chunking breaks a large document into smaller parts. These smaller parts are easier to process and can later help a retrieval system select only the relevant section. In this assignment, splitting `notes.txt` lets me work with smaller pieces instead of treating the complete file as one large block.

## Task 7 - Length-Based Text Splitter

I use `CharacterTextSplitter` with a chunk size of 300 and overlap of 50. The same text document loaded in Task 1 is used here.

In [7]:
character_splitter = CharacterTextSplitter(
    separator="\n\n",
    chunk_size=300,
    chunk_overlap=50,
)

character_chunks = character_splitter.split_documents(text_documents)

print("Chunk size:", 300)
print("Chunk overlap:", 50)
print("Number of chunks:", len(character_chunks))

print("\nSample chunk:")
print(character_chunks[0].page_content)

print("\nChunk lengths:")
print([len(chunk.page_content) for chunk in character_chunks])

Created a chunk of size 377, which is longer than the specified 300
Created a chunk of size 560, which is longer than the specified 300
Created a chunk of size 429, which is longer than the specified 300


Chunk size: 300
Chunk overlap: 50
Number of chunks: 6

Sample chunk:
Personal Knowledge Assistant - Project Notes

Chunk lengths:
[44, 377, 560, 429, 275, 235]


### Observation

The chunk count and chunk lengths above show how `CharacterTextSplitter` divided the loaded notes using paragraph breaks as the separator. The 50-character overlap is configured to keep some context between nearby chunks.

## Task 8 - Text Structure-Based Splitter

`RecursiveCharacterTextSplitter` tries different text boundaries instead of depending on only one separator. It first tries larger natural boundaries and falls back to smaller ones when needed. This helps it keep paragraphs and words together where possible.

I apply it to the same `notes.txt` document and compare it with Task 7.

In [8]:
recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50,
)

recursive_chunks = recursive_splitter.split_documents(text_documents)

print("CharacterTextSplitter chunks:", len(character_chunks))
print("RecursiveCharacterTextSplitter chunks:", len(recursive_chunks))

print("\nCharacter splitter chunk lengths:")
print([len(chunk.page_content) for chunk in character_chunks])

print("\nRecursive splitter chunk lengths:")
print([len(chunk.page_content) for chunk in recursive_chunks])

print("\nSample CharacterTextSplitter chunk:")
print(character_chunks[0].page_content)

print("\nSample RecursiveCharacterTextSplitter chunk:")
print(recursive_chunks[0].page_content)

CharacterTextSplitter chunks: 6
RecursiveCharacterTextSplitter chunks: 10

Character splitter chunk lengths:
[44, 377, 560, 429, 275, 235]

Recursive splitter chunk lengths:
[44, 258, 118, 256, 228, 74, 245, 183, 275, 235]

Sample CharacterTextSplitter chunk:
Personal Knowledge Assistant - Project Notes

Sample RecursiveCharacterTextSplitter chunk:
Personal Knowledge Assistant - Project Notes


### Comparison

The output above gives a direct comparison using the same source document, `chunk_size=300`, and `chunk_overlap=50`. I can compare both the number of chunks and their actual lengths. The recursive splitter can fall back to smaller separators when a paragraph is too large, while `CharacterTextSplitter` only uses the separator I configured.

## Task 9 - Document Structure-Based Splitting

For structured content, I use `MarkdownHeaderTextSplitter`. The sample Markdown has headings and subheadings. The splitter stores the heading hierarchy in metadata, which helps preserve section information.

In [9]:
markdown_text = """# Company Overview

## Engineering Practices
Our engineering team follows clean code principles and a defined Software Development Life Cycle.
Code reviews and automated testing are required before merging code.

## Tooling
Developers use VS Code and Azure DevOps for development and work tracking.

## Onboarding

### Technical Training
New employees complete Python, React, and Power Platform training modules.
"""

headers_to_split_on = [
    ("#", "Main Section"),
    ("##", "Section"),
    ("###", "Subsection"),
]

markdown_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=headers_to_split_on
)

structure_chunks = markdown_splitter.split_text(markdown_text)

print("Number of structured chunks:", len(structure_chunks))

for index, chunk in enumerate(structure_chunks, start=1):
    print(f"\n--- Chunk {index} ---")
    print("Metadata:", chunk.metadata)
    print("Content:", chunk.page_content)

Number of structured chunks: 3

--- Chunk 1 ---
Metadata: {'Main Section': 'Company Overview', 'Section': 'Engineering Practices'}
Content: Our engineering team follows clean code principles and a defined Software Development Life Cycle.
Code reviews and automated testing are required before merging code.

--- Chunk 2 ---
Metadata: {'Main Section': 'Company Overview', 'Section': 'Tooling'}
Content: Developers use VS Code and Azure DevOps for development and work tracking.

--- Chunk 3 ---
Metadata: {'Main Section': 'Company Overview', 'Section': 'Onboarding', 'Subsection': 'Technical Training'}
Content: New employees complete Python, React, and Power Platform training modules.


### Observation

The sample chunks show the section names in metadata. This is different from only splitting by character length because the document's heading structure is preserved.

## Task 10 - Semantic Meaning-Based Splitting

### 1. What does semantic chunking mean?

Semantic chunking splits text when the meaning or topic changes instead of splitting only after a fixed number of characters.

For example, `notes.txt` discusses document loaders and chunking. A semantic splitter would try to keep the loader discussion together and create another chunk when the topic changes to text chunking.

### 2. How do embeddings help semantic splitting?

Embeddings convert text into numerical vectors that represent meaning. Text with similar meaning usually has similar vectors. A semantic splitter can compare nearby sentences or groups of sentences. A large change in similarity can indicate a topic change and a possible chunk boundary.

### Optional Demo

I did not add a semantic splitter demo because the assignment marks it as optional. The main implementation stays focused on LangChain document loaders and the required standard text splitters.

# Part 3 - Mini Integration Task

## Task 11 - Unified Preprocessing Pipeline

The function below accepts either a local directory or a web URL. It loads the documents, splits them with `RecursiveCharacterTextSplitter`, and returns the chunks.

In [10]:
LOADER_MAP = {
    ".txt": (TextLoader, {"encoding": "utf-8"}),
    ".csv": (CSVLoader, {}),
    ".pdf": (PyPDFLoader, {}),
}


def load_and_split_documents(path_or_url):
    parsed_url = urlparse(path_or_url)

    if parsed_url.scheme in ("http", "https"):
        loader = WebBaseLoader(path_or_url)
        documents = loader.load()

    elif os.path.isdir(path_or_url):
        documents = []

        for extension, (loader_class, loader_kwargs) in LOADER_MAP.items():
            loader = DirectoryLoader(
                path_or_url,
                glob=f"**/*{extension}",
                loader_cls=loader_class,
                loader_kwargs=loader_kwargs,
            )
            documents.extend(loader.load())

    else:
        raise ValueError("Please provide a valid local directory or web URL.")

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=300,
        chunk_overlap=50,
    )

    chunks = splitter.split_documents(documents)
    return chunks

### Apply the pipeline to the local directory

In [11]:
local_chunks = load_and_split_documents("data")

print("Chunks returned from local directory:", len(local_chunks))
print("Sources included:")

for source in sorted(set(chunk.metadata.get("source", "Unknown") for chunk in local_chunks)):
    print("-", source)

print("\nSample local chunk:")
print(local_chunks[0].page_content[:500])

Chunks returned from local directory: 22
Sources included:
- data\company_overview.pdf
- data\data.csv
- data\notes.txt

Sample local chunk:
Personal Knowledge Assistant - Project Notes


### Apply the pipeline to a web URL

**Note:** This cell requires an internet connection.

In [12]:
web_chunks = load_and_split_documents("https://www.example.com/")

print("Chunks returned from web URL:", len(web_chunks))
print("\nSample web chunk:")
print(web_chunks[0].page_content[:500])

Chunks returned from web URL: 1

Sample web chunk:
Example DomainExample DomainThis domain is for use in documentation examples without needing permission. Avoid use in operations.Learn more


### Observation

The same function can process two different input types. For a local directory, it uses the suitable loader for each supported file extension. For a URL, it uses `WebBaseLoader`. Both paths use the same recursive splitter before returning chunks.

## Task 12 - Observations & Insights

### 1. Which loader is used for which data type?

| Data Type | Loader |
|---|---|
| TXT | `TextLoader` |
| CSV | `CSVLoader` |
| PDF | `PyPDFLoader` |
| Folder / directory | `DirectoryLoader` with a suitable loader class |
| Public webpage | `WebBaseLoader` |

### 2. Best splitter for different data

**Small text:**
For simple small text, `CharacterTextSplitter` can be enough when I only need basic length-based splitting.

**Large PDFs:**
I would prefer `RecursiveCharacterTextSplitter` after loading the PDF because it can try smaller text boundaries when a large section does not fit the target chunk size. PDF page metadata can also help trace chunks back to their source pages.

**Web data:**
I would use `RecursiveCharacterTextSplitter` for general extracted web text. If the content has reliable Markdown headings, a header-based splitter can preserve those sections.

### 3. Why is chunk overlap important?

Chunk overlap repeats a small amount of text between nearby chunks. This is useful when an idea is close to a chunk boundary. Without overlap, related context may be separated. In Task 7 and Task 8, I configured an overlap of 50 characters so nearby chunks can retain some shared context.

# Final Learning Reflection

From this assignment, I understood that different data formats need suitable loaders, but LangChain converts them into a common `Document` format with `page_content` and `metadata`.

I also understood that loading and splitting are separate steps. A loader reads the source, while a text splitter divides the loaded documents into smaller chunks. `CharacterTextSplitter` follows the separator I provide, while `RecursiveCharacterTextSplitter` can try smaller boundaries when needed.

The unified function in Task 11 helped me connect both steps into one preprocessing flow.

# Challenges Faced

No personal challenge is claimed in this notebook yet. If an installation, file path, or internet error occurs while running the notebook, I will document the actual issue and the fix instead of adding a generic challenge.